In [2]:
import pandas as pd
import numpy as np

In [7]:
# import pandas as pd
# import numpy as np

dp04_cols = ['GEO_ID', 'NAME', 'DP04_0001E', 'DP04_0007E',  # housing characteristics: Total households (DP04_0001E), Owner Occupied (DP04_0007E)
             # Housing age columns (20+ years = 2004 or earlier)
             # 'DP04_0029E', 'DP04_0030E', 'DP04_0031E',  # Recent (exclude these)
             'DP04_0032E', 'DP04_0033E', 'DP04_0034E',  # 1970-1999
             'DP04_0035E', 'DP04_0036E', 'DP04_0037E'   # 1939-1969 (oldest)
            ]
dp03_cols = ['GEO_ID', 'DP03_0062E']                       # median incomes
dp02_cols = [
    'GEO_ID',                  # Geographical ID
    'DP02_0001E',              # Total households  
    'DP02_0008E',              # 4-person households
    'DP02_0009E',              # 5-person households
    'DP02_0010E',              # 6-person households
    'DP02_0011E'               # 7-or-more person households
] # social charactersitics

# Load datasets using above columns
dp04 = pd.read_csv('Datasets/ACSDP5Y2023.DP04-Data.csv', usecols=dp04_cols, low_memory=False)
dp03 = pd.read_csv('Datasets/ACSDP5Y2023.DP03-Data.csv', usecols=dp03_cols, low_memory=False)
dp02 = pd.read_csv('Datasets/ACSDP5Y2023.DP02-Data.csv', usecols=dp02_cols, low_memory=False)

# Merge on GEO_ID 
df = dp04.merge(dp03, on='GEO_ID').merge(dp02, on='GEO_ID')

# Convert to numeric
numeric_cols = ['DP04_0001E', 'DP04_0007E', 'DP03_0062E', 'DP02_0001E', 
                'DP02_0008E', 'DP02_0009E', 'DP02_0010E', 'DP02_0011E',
                'DP04_0032E', 'DP04_0033E', 'DP04_0034E',  'DP04_0035E', 
                'DP04_0036E', 'DP04_0037E']
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Calculate metrics
df['large_households'] = (df['DP02_0008E'] + df['DP02_0009E'] + 
                         df['DP02_0010E'] + df['DP02_0011E'])
df['pct_large_households'] = df['large_households'] / df['DP02_0001E']
df['homeownership_rate'] = df['DP04_0007E'] / df['DP04_0001E']
older_housing_cols = ['DP04_0032E','DP04_0034E','DP04_0035E','DP04_0036E','DP04_0037E']  # 1999 and earlier
df['pct_old_housing'] = df[older_housing_cols].sum(axis=1) / df['DP04_0001E']


# Filter criteria
filtered = df[
    (df['homeownership_rate'] > 0.6) &
    (df['DP03_0062E'] > 75000) &
    (df['pct_large_households'] > 0.25) &
    (df['pct_old_housing'] > 0.5)  # >50% houses over 25 years old
].copy()

# Output
result = filtered[['GEO_ID', 'NAME', 'homeownership_rate', 'DP03_0062E', 'pct_large_households', 'pct_old_housing']].round(3)
result.columns = ['GEO_ID', 'Geography', 'Homeownership%', 'MedianIncome', 'LargeHH%', 'OldHouse%']

print(f"Found {len(result)} matching geographies:")
print(result.head(10))
result.to_csv('Datasets/busy_families_housing.csv', index=False)


Found 8062 matching geographies:
             GEO_ID    Geography  Homeownership%  MedianIncome  LargeHH%  \
136  860Z200US01005  ZCTA5 01005           0.770       97390.0     0.315   
137  860Z200US01007  ZCTA5 01007           0.733       99056.0     0.431   
138  860Z200US01008  ZCTA5 01008           0.936       90625.0     0.456   
140  860Z200US01010  ZCTA5 01010           0.884      101477.0     0.309   
141  860Z200US01011  ZCTA5 01011           0.884       87012.0     0.287   
142  860Z200US01012  ZCTA5 01012           0.822       75526.0     0.561   
146  860Z200US01026  ZCTA5 01026           0.837       98438.0     0.422   
148  860Z200US01028  ZCTA5 01028           0.856      100997.0     0.445   
149  860Z200US01029  ZCTA5 01029           0.986       99188.0     0.534   
150  860Z200US01030  ZCTA5 01030           0.778       92485.0     0.398   

     OldHouse%  
136      0.675  
137      0.576  
138      0.722  
140      0.601  
141      0.656  
142      0.565  
146      0.

In [25]:
result.isna().sum()

GEO_ID            0
Geography         0
Homeownership%    0
MedianIncome      0
LargeHH%          0
dtype: int64

In [26]:
df.shape

(33773, 13)

In [28]:
dp04.shape

(33773, 4)